****Imports & Environment****

here we put all of our imports.

Optional libraries are imported with try/except blocks so the script still
runs even if they are missing:
- albumentations  → richer augmentation pipeline
- sklearn         → AUC metric + Platt calibration
- torch._dynamo   → torch.compile support

ImageNet mean and std constants are defined here for normalisation.


In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import gc
import copy
import time
import argparse
import random
import cv2
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from PIL import Image
from tqdm import tqdm
from torch.optim.swa_utils import AveragedModel

cv2.setNumThreads(0)

_CROPPED_DIR     = "/kaggle/input/datasets/mohameddhiabenothman/dataset-cropped-26-v6/dataset_cropped"
_KAGGLE_CSV_FILE = "/kaggle/input/competitions/1st-krones-vision-ai-challenge/train.csv"
_KAGGLE_OUT      = "/kaggle/working/assets/model_v6.pt"
_RESUME_CKPT     = "/kaggle/working/assets/model_v6.pt"
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    _ALBU_OK = True
except ImportError:
    _ALBU_OK = False
    print("[WARN] albumentations not found — using torchvision transforms.")

try:
    from sklearn.metrics import roc_auc_score
    from sklearn.linear_model import LogisticRegression
    _SKLEARN_OK = True
except ImportError:
    _SKLEARN_OK = False

try:
    import torch._dynamo
    _DYNAMO_OK = True
except ImportError:
    _DYNAMO_OK = False

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

****CONFIGURATION & CLI ARGUMENTS****

What it does:
Defines all hyperparameters and file paths using argparse. On Kaggle,
args = parser.parse_args(args=[]) is used so defaults are applied
without a command line.

Paths:
--dataset       : folder of pre-cropped training images
--csv           : CSV with image_id and target columns
--out           : where to save the best checkpoint (.pt)
--resume        : checkpoint to resume training from

Training schedule:
--phase1-epochs : epochs to train head only (default 5)
--phase2-epochs : epochs to train head + last blocks (default 8)
--phase3-epochs : full model fine-tuning (default 30)
--swa-start     : epoch to begin Stochastic Weight Averaging (default 30)
--early-stop    : patience for early stopping (default 12)

Hyperparameters:
--batch         : batch size (default 8 - reduced for VRAM)
--lr            : base learning rate (default 5e-5)
--img-size      : input image size in pixels (default 260)
--label-smooth  : label smoothing epsilon (default 0.02)
--mixup-alpha   : MixUp/CutMix blend strength (default 0.1)
--focal-gamma   : Focal Loss focusing parameter (default 2.0)
--ema-decay     : EMA smoothing factor (default 0.9995)
--grad-clip     : gradient clipping max norm (default 0.5)
--accum-steps   : gradient accumulation steps (default 16 - compensates for batch size 8)
--val-split     : percentage of data used for validation (default 0.12)
--dropout-rate  : dropout applied in model head (default 0.35)

Flags:
--bf16          : use bfloat16 AMP on Ampere+ GPUs (default True)
--channels-last : NCHW → NHWC memory layout for GPU speedup (default True)
--compile       : torch.compile for max-autotune (default False)
--oversample    : duplicate minority class samples (default True)


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument("--dataset",            default=_CROPPED_DIR)
parser.add_argument("--csv",                default=_KAGGLE_CSV_FILE)
parser.add_argument("--out",                default=_KAGGLE_OUT)
parser.add_argument("--resume",             default=_RESUME_CKPT)
parser.add_argument("--phase1-epochs",      type=int,   default=5)
parser.add_argument("--phase2-epochs",      type=int,   default=8)
parser.add_argument("--phase3-epochs",      type=int,   default=27)
parser.add_argument("--swa-start",          type=int,   default=30)
parser.add_argument("--early-stop",         type=int,   default=12)
parser.add_argument("--batch",              type=int,   default=16)
parser.add_argument("--lr",                 type=float, default=5e-5)
parser.add_argument("--img-size",           type=int,   default=260)
parser.add_argument("--workers",            type=int,   default=4)
parser.add_argument("--seed",               type=int,   default=42)
parser.add_argument("--val-split",          type=float, default=0.12)
parser.add_argument("--test-split",         type=float, default=0.00)
parser.add_argument("--label-smooth",       type=float, default=0.02)
parser.add_argument("--mixup-alpha",        type=float, default=0.1)
parser.add_argument("--focal-gamma",        type=float, default=2.0)
parser.add_argument("--ema-decay",          type=float, default=0.9998)
parser.add_argument("--grad-clip",          type=float, default=0.5)
parser.add_argument("--accum-steps",        type=int,   default=8)   # batch*accum=128 effective
parser.add_argument("--bn-recal-batches",   type=int,   default=25)
# TTA: freq=0 means no TTA during training, only at final eval
parser.add_argument("--tta-freq",           type=int,   default=0)
parser.add_argument("--tta-views-final",    type=int,   default=16)
parser.add_argument("--val-freq",           type=int,   default=1)
parser.add_argument("--oversample",         action="store_true", default=True)
parser.add_argument("--no-oversample",      action="store_false", dest="oversample")
parser.add_argument("--grad-checkpoint",    action="store_true", default=False)
# bf16 only used on Ampere+ — disabled on T4 automatically
parser.add_argument("--bf16",               action="store_true", default=True)
parser.add_argument("--no-bf16",            action="store_false", dest="bf16")
parser.add_argument("--channels-last",      action="store_true", default=True)
parser.add_argument("--no-channels-last",   action="store_false", dest="channels_last")
parser.add_argument("--calibrate",          action="store_true", default=True)
parser.add_argument("--no-calibrate",       action="store_false", dest="calibrate")
parser.add_argument("--compile",            action="store_true", default=False)
parser.add_argument("--dropout-rate",       type=float, default=0.35)
parser.add_argument("--multi-scale",        action="store_true", default=False)
parser.add_argument("--cos-restart-period", type=int,   default=7)
parser.add_argument("--cache-val",          action="store_true", default=True)
parser.add_argument("--no-cache-val",       action="store_false", dest="cache_val")
parser.add_argument("--profile-val",        action="store_true", default=False)

args = parser.parse_args(args=[])
args.epochs = args.phase1_epochs + args.phase2_epochs + args.phase3_epochs
if args.img_size == 260: 
    args.dataset = "/kaggle/input/datasets/mohameddhiabenothman/dataset-cropped-26-v6/dataset_cropped"
else: 
    args.dataset = "/kaggle/input/datasets/mohameddhiabenothman/dataset-cropped/dataset_cropped_v6"

****CUSTOM AUGMENTATION CLASSES****

These are domain-specific augmentations written as Albumentations
ImageOnlyTransform subclasses. They simulate real-world visual noise
specific to industrial rim inspection images.

**RadialBlur:**
The "Why": Rims are circular and often rotating on a conveyor belt or spindle when photographed. This movement causes a specific type of directional motion blur.

The "How": It creates a custom matrix (a diagonal convolution kernel) and applies it using cv2.filter2D. This smears the pixel values along a specific axis, forcing the model to learn what a defect looks like even if the camera shutter was slightly too slow.

**SpecularHighlight:** 
The "Why": Metal is highly reflective. Factory lighting will inevitably cast bright, blinding white glares (specular highlights) on the rim, which can wash out or hide defects.

The "How": It calculates a 2D Gaussian mask (an elliptical shape where the center is intense and it fades at the edges) and aggressively adds brightness to the image. It trains the model not to be fooled by sudden bright spots.

**ScratchSimulation:** 
The "Why": Rims get scratched in transit. A shallow surface scratch is usually fine, but a deep structural crack is a critical defect. The model must learn to distinguish between the two.

The "How": It uses basic trigonometry (np.sin, np.cos) to draw random, near-white lines (cv2.line) of varying lengths and angles across the image, acting as "fake" harmless scratches.

**SectorMask:** 
The "Why": Sometimes, parts of the machine holding the rim, or shadows, block a "slice" of the camera's view.

The "How": It calculates the average color of the image and uses cv2.ellipse to draw a solid pie-slice shape over the rim. This forces the model to make a prediction based on the visible parts of the rim, preventing it from crashing or heavily skewing its prediction if a piece of the image is missing.

**RimSpecificAug:** 
The "Why": This simulates the physical setup of the camera rig.

The "How": It randomly chooses one of three specific environmental noises:

Vignette: Darkens the corners to simulate cheap camera lenses or specific spotlighting.

Ring: Adds a faint circular overlay, simulating the reflection of a "ring light" (a very common lighting setup in macro/industrial photography).

Fixture: Draws a solid block on one edge, mimicking a mounting bracket or robotic arm holding the rim in place.



In [3]:
class RadialBlur(A.ImageOnlyTransform if _ALBU_OK else object):
    """Single filter2D call instead of multiple warpAffine loops."""
    def __init__(self, max_strength=6, p=0.3):
        if _ALBU_OK:
            super().__init__(p=p)
        self.max_strength = max_strength

    def apply(self, img, **params):
        strength    = random.randint(2, self.max_strength)
        kernel_size = strength * 2 + 1
        kernel      = np.zeros(
            (kernel_size, kernel_size), dtype=np.float32)
        for i in range(kernel_size):
            kernel[i, i] = 1.0 / kernel_size
        return cv2.filter2D(img, -1, kernel)

    def get_transform_init_args_names(self):
        return ("max_strength",)


class SpecularHighlight(A.ImageOnlyTransform if _ALBU_OK else object):
    def __init__(self, p=0.3):
        if _ALBU_OK:
            super().__init__(p=p)

    def apply(self, img, **params):
        h, w      = img.shape[:2]
        result    = img.copy().astype(np.float32)
        cx        = random.randint(w // 4, 3 * w // 4)
        cy        = random.randint(h // 4, 3 * h // 4)
        rx        = random.randint(w // 10, w // 4)
        ry        = random.randint(h // 10, h // 4)
        y_idx, x_idx = np.ogrid[:h, :w]
        mask      = np.exp(-(((x_idx - cx) / rx) ** 2 +
                              ((y_idx - cy) / ry) ** 2))
        intensity = random.uniform(0.2, 0.7)
        for c in range(3):
            result[:, :, c] += mask * intensity * 255
        return np.clip(result, 0, 255).astype(np.uint8)

    def get_transform_init_args_names(self):
        return ()


class ScratchSimulation(A.ImageOnlyTransform if _ALBU_OK else object):
    def __init__(self, max_scratches=4, p=0.25):
        if _ALBU_OK:
            super().__init__(p=p)
        self.max_scratches = max_scratches

    def apply(self, img, **params):
        result = img.copy()
        h, w   = result.shape[:2]
        for _ in range(random.randint(1, self.max_scratches)):
            x1        = random.randint(0, w)
            y1        = random.randint(0, h)
            length    = random.randint(20, w // 2)
            angle_rad = random.uniform(0, np.pi)
            x2        = int(x1 + length * np.cos(angle_rad))
            y2        = int(y1 + length * np.sin(angle_rad))
            color     = tuple(random.randint(180, 255) for _ in range(3))
            cv2.line(result, (x1, y1), (x2, y2),
                     color, random.randint(1, 2))
        return result

    def get_transform_init_args_names(self):
        return ("max_scratches",)


class SectorMask(A.ImageOnlyTransform if _ALBU_OK else object):
    def __init__(self, max_angle=60.0, p=0.2):
        if _ALBU_OK:
            super().__init__(p=p)
        self.max_angle = max_angle

    def apply(self, img, **params):
        result     = img.copy()
        h, w       = result.shape[:2]
        cx, cy     = w // 2, h // 2
        start_ang  = random.uniform(0, 360)
        sweep_ang  = random.uniform(10, self.max_angle)
        mean_color = tuple(int(c) for c in img.mean(axis=(0, 1)))
        radius     = int(np.sqrt((w / 2) ** 2 + (h / 2) ** 2)) + 1
        cv2.ellipse(result, (cx, cy), (radius, radius),
                    0, start_ang, start_ang + sweep_ang,
                    mean_color, -1)
        return result

    def get_transform_init_args_names(self):
        return ("max_angle",)


class RimSpecificAug(A.ImageOnlyTransform if _ALBU_OK else object):
    def __init__(self, p=0.3):
        if _ALBU_OK:
            super().__init__(p=p)

    def apply(self, img, **params):
        h, w   = img.shape[:2]
        result = img.copy().astype(np.float32)
        mode   = random.choice(["vignette", "ring", "fixture"])

        if mode == "vignette":
            cx, cy       = w / 2, h / 2
            y_idx, x_idx = np.ogrid[:h, :w]
            dist         = np.sqrt(((x_idx - cx) / cx) ** 2 +
                                    ((y_idx - cy) / cy) ** 2)
            strength = random.uniform(0.3, 0.7)
            vignette = 1.0 - np.clip(dist * strength, 0, 0.6)
            result  *= vignette[:, :, np.newaxis]

        elif mode == "ring":
            cx, cy       = w // 2, h // 2
            y_idx, x_idx = np.ogrid[:h, :w]
            dist         = np.sqrt((x_idx - cx) ** 2 + (y_idx - cy) ** 2)
            r1   = random.uniform(0.2, 0.4) * min(w, h)
            r2   = r1 + random.uniform(5, 20)
            ring = ((dist >= r1) & (dist <= r2)).astype(np.float32)
            tint = np.array([random.uniform(0.8, 1.2) for _ in range(3)])
            result += ring[:, :, np.newaxis] * tint * 30

        else:
            side  = random.choice(["top", "bottom", "left", "right"])
            thick = random.randint(h // 12, h // 6)
            color = np.array([random.uniform(100, 200) for _ in range(3)])
            if side == "top":
                result[:thick, :]    = color
            elif side == "bottom":
                result[h - thick:, :] = color
            elif side == "left":
                result[:, :thick]    = color
            else:
                result[:, w - thick:] = color

        return np.clip(result, 0, 255).astype(np.uint8)

    def get_transform_init_args_names(self):
        return ()

****IMAGE TRANSFORMS****

This section organizes how and when the model experiences the augmentations. It uses a strategy called curriculum learning—starting easy, and getting progressively harder.

***heavy=False (The Warm-Up Phase):***

Used in training Phases 1 and 2.

When the model's new classification head is first initialized, its weights are totally random. If you immediately feed it images covered in heavy glare, scratches, and missing sectors, the model will fail to learn the basic patterns of a rim.

Instead, it only uses basic augmentations: 90-degree rotations, flips, minor brightness tweaks, and a bit of Gaussian noise. It's just enough to prevent immediate overfitting while the model learns the core task.

***heavy=True (The "Kitchen Sink" Phase):***

Used in training Phase 3.

Once the model understands the basics, the training wheels come off. This pipeline stacks all the custom augmentations from Section 3 alongside advanced Albumentations functions:

ShiftScaleRotate & ElasticTransform: Warps and stretches the image, simulating lens distortion or variations in the distance between the camera and the rim.

CLAHE: (Contrast Limited Adaptive Histogram Equalization). This is a highly effective algorithm for industrial greyscale images; it locally enhances contrast so hidden textures pop out.

CoarseDropout: Literally cuts random black squares out of the image. This prevents the model from hyper-fixating on one single pixel or feature, forcing it to look at the entire context of the rim.

Validation Transform:

Unlike training, validation uses zero augmentations. It strictly resizes the image and normalizes the pixel values.

Validation needs to be a stable, deterministic benchmark. If you augment your validation set, your accuracy scores will bounce up and down randomly, making it impossible to tell if your model is actually getting smarter or just getting lucky with the augmentation dice rolls.



In [4]:
def _make_train_transform(img_size, heavy=True):
    """
    heavy=False → light augmentation (phases 1 & 2) — faster CPU
    heavy=True  → full augmentation  (phase 3)
    """
    if _ALBU_OK:
        if not heavy:
            return A.Compose([
                A.RandomRotate90(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.RandomBrightnessContrast(
                    brightness_limit=0.2,
                    contrast_limit=0.2, p=0.4),
                A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
                A.Normalize(mean=IMAGENET_MEAN.tolist(),
                            std=IMAGENET_STD.tolist()),
                ToTensorV2(),
            ])
        return A.Compose([
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.ShiftScaleRotate(
                shift_limit=0.05, scale_limit=0.10,
                rotate_limit=15, p=0.5,
                border_mode=cv2.BORDER_REFLECT_101),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.OneOf([
                A.RandomBrightnessContrast(
                    brightness_limit=0.25, contrast_limit=0.25),
                A.HueSaturationValue(
                    hue_shift_limit=10, sat_shift_limit=20,
                    val_shift_limit=15),
                A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8)),
                A.RandomGamma(gamma_limit=(80, 120)),
            ], p=0.7),
            A.OneOf([
                A.GaussNoise(var_limit=(5.0, 30.0)),
                A.ISONoise(color_shift=(0.01, 0.04),
                           intensity=(0.05, 0.3)),
                A.MultiplicativeNoise(multiplier=(0.95, 1.05)),
            ], p=0.35),
            A.OneOf([
                A.MotionBlur(blur_limit=5),
                A.GaussianBlur(blur_limit=(3, 5)),
                A.MedianBlur(blur_limit=3),
            ], p=0.25),
            A.CoarseDropout(
                max_holes=6,
                max_height=img_size // 14,
                max_width=img_size // 14,
                p=0.35),
            RadialBlur(p=0.25),
            SpecularHighlight(p=0.25),
            ScratchSimulation(p=0.20),
            SectorMask(p=0.20),
            RimSpecificAug(p=0.30),
            A.Normalize(mean=IMAGENET_MEAN.tolist(),
                        std=IMAGENET_STD.tolist()),
            ToTensorV2(),
        ])
    else:
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.25, contrast=0.25,
                          saturation=0.25, hue=0.08),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN.tolist(),
                        std=IMAGENET_STD.tolist()),
        ])


def _make_val_transform():
    if _ALBU_OK:
        return A.Compose([
            A.Normalize(mean=IMAGENET_MEAN.tolist(),
                        std=IMAGENET_STD.tolist()),
            ToTensorV2(),
        ])
    else:
        return T.Compose([
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN.tolist(),
                        std=IMAGENET_STD.tolist()),
        ])

**DATASET CLASS**

This section defines the blueprint for how PyTorch interacts with your raw image files. The Dataset class acts as the "factory worker" that retrieves a single image and its label whenever the neural network requests one.

**Caching (cache=True/False):** 
Reading images directly from a hard drive (Disk I/O) is painfully slow. If your dataset has 10,000 images and you train for 30 epochs, you are reading from the disk 300,000 times.

When cache=True, the **_preload()** function reads all images once at the start of the script and stores them as NumPy arrays directly in your system's RAM.

The script uses this specifically for the Validation Set. Because the validation set doesn't use random augmentations (it needs to be identical every time to provide a stable benchmark), it's incredibly efficient to just hold it in RAM.

**Multi-Scale (multi_scale=True):** Instead of resizing every image to exactly 260x260 pixels, it randomly picks from three sizes: 85% (221px), 100% (260px), and 115% (299px). This forces the model to become "scale-invariant," meaning it learns to recognize a defect whether the camera was positioned slightly closer or slightly further away from the rim.(it is an additional security to insure the model detects most of the images)

**__getitem__(self, idx):** This is the core engine of the class. PyTorch says, "Give me item number 42." This function:

Grabs image 42 from RAM (or reads it from disk).

Resizes it based on the multi-scale rule.

Pushes it through the augmentation pipeline (albumentations).

Converts it into a PyTorch Tensor (the mathematical matrix the GPU understands) and returns it alongside its true label (0 for good, 1 for faulty).

In [5]:
class RimDataset(Dataset):
    """
    cache=True  → preload all images into RAM (recommended for val set)
    cache=False → read from disk each time
    """
    def __init__(self, samples, transform,
                 img_size=300, multi_scale=False,
                 cache=False):
        self.samples     = samples
        self.transform   = transform
        self.img_size    = img_size
        self.multi_scale = multi_scale
        self.cache       = cache
        self._img_cache  = {}
        self.scales      = [
            int(img_size * 0.85),
            img_size,
            int(img_size * 1.15),
        ]
        if cache:
            self._preload()

    def _preload(self):
        print(f"  [Cache] Loading {len(self.samples)}"
              f" images into RAM…", flush=True)
        for idx, (path, _) in enumerate(
                tqdm(self.samples, desc="  caching",
                     leave=False)):
            self._img_cache[idx] = \
                self._read_img(path, self.img_size)
        print("  [Cache] Done.", flush=True)

    def _read_img(self, path, size):
        img = Image.open(path).convert("RGB")
        if img.size != (size, size):
            img = img.resize((size, size), Image.BILINEAR)
        return np.array(img)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        if self.cache and idx in self._img_cache:
            img_arr = self._img_cache[idx]
        else:
            size    = (random.choice(self.scales)
                       if self.multi_scale
                       else self.img_size)
            img_arr = self._read_img(path, size)
            if img_arr.shape[:2] != (self.img_size,
                                      self.img_size):
                img_arr = np.array(
                    Image.fromarray(img_arr).resize(
                        (self.img_size, self.img_size),
                        Image.BILINEAR))

        if _ALBU_OK:
            tensor = self.transform(image=img_arr)["image"]
        else:
            tensor = self.transform(Image.fromarray(img_arr))

        return tensor, torch.tensor(label, dtype=torch.float32)


**DATA LOADING — SAMPLING, SPLITTING, DATALOADERS**


If the Dataset is the factory worker grabbing individual items, the DataLoader is the conveyor belt that groups them into batches and ships them to the GPU.

**Stratified Splitting (_split_samples):**
If 90% of your rims are perfect and 10% are defective, a completely random shuffle might accidentally put zero defective rims into your validation set. Stratified splitting manually separates the "good" and "faulty" images first, calculates the math, and ensures that the 90/10 ratio is perfectly preserved across your Training, Validation, and Test sets.

**Handling Class Imbalance (_oversample_minority vs _make_sampler):** 
Neural networks are lazy. If 90% of the dataset is "good," the model will realize it can achieve 90% accuracy just by guessing "good" every single time, without actually learning anything. You have to force it to care about the defects.

**Oversampling:** 
The script calculates the exact difference between the two classes and randomly duplicates the defective images in the list until the ratio is exactly 1:1.

**Weighted Random Sampler:**
Alternatively, instead of duplicating data, it assigns a mathematical weight (probability) to each sample. Defective samples get a high weight, meaning they are pulled more frequently from the dataset pool.

**DataLoaders (_make_loaders):** Kaggle Worker Limit: I did explicitly force num_workers = min(max(2, args.workers), 4). Kaggle notebooks are notorious for crashing if you spawn too many CPU threads to fetch data. Capping it at 2 to 4 workers ensures the notebook stays alive. (this detail is just for the notebook)

**pin_memory=True:** This is a PyTorch trick that allocates a special block of CPU memory that is directly linked to the GPU, making the transfer of data batches incredibly fast.

In [6]:
def _load_samples(cropped_dir, csv_path):
    root     = Path(cropped_dir)
    csv_file = Path(csv_path)

    if not csv_file.exists():
        raise FileNotFoundError(f"CSV not found: {csv_file}")
    if not root.exists():
        raise FileNotFoundError(f"Cropped dir not found: {root}")

    lookup = {}
    for p in root.iterdir():
        if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".tiff"):
            lookup[p.stem] = p
            lookup[p.name] = p

    df = pd.read_csv(csv_file)
    for col in ("image_id", "target"):
        if col not in df.columns:
            raise ValueError(f"CSV missing column '{col}'.")

    samples, missing = [], 0
    for _, row in df.iterrows():
        img_id = str(row["image_id"])
        found  = lookup.get(img_id) or lookup.get(Path(img_id).stem)
        if found:
            samples.append((str(found), int(row["target"])))
        else:
            missing += 1

    if missing:
        print(f"[WARN] {missing} image(s) not matched — skipped.")
    if not samples:
        raise RuntimeError("No samples matched.")
    print(f"[INFO] Loaded {len(samples)} samples.")
    return samples


def _split_samples(samples, val_frac, test_frac, seed=42):
    rng    = random.Random(seed)
    good   = [s for s in samples if s[1] == 0]
    faulty = [s for s in samples if s[1] == 1]
    rng.shuffle(good)
    rng.shuffle(faulty)

    def _split(lst):
        nv = max(1, int(len(lst) * val_frac))
        nt = int(len(lst) * test_frac) if test_frac > 0 else 0
        return lst[nv + nt:], lst[:nv], lst[nv:nv + nt]

    tr_g, vl_g, te_g = _split(good)
    tr_f, vl_f, te_f = _split(faulty)
    train = tr_g + tr_f
    rng.shuffle(train)

    n0 = sum(1 for _, l in train if l == 0)
    n1 = sum(1 for _, l in train if l == 1)
    print(f"[INFO] Split → Train:{len(train)}"
          f"  Val:{len(vl_g)+len(vl_f)}"
          f"  Test:{len(te_g)+len(te_f)}")
    print(f"[INFO] Balance → Good:{n0}  Faulty:{n1}"
          f"  Ratio:{n0/max(n1,1):.2f}:1")
    return train, vl_g + vl_f, te_g + te_f


def _oversample_minority(train_samples, seed=42, target_ratio=1.0):
    rng      = random.Random(seed)
    good     = [s for s in train_samples if s[1] == 0]
    faulty   = [s for s in train_samples if s[1] == 1]
    if len(good) == len(faulty):
        return train_samples
    minority = faulty if len(faulty) < len(good) else good
    majority = good   if len(faulty) < len(good) else faulty
    n_extra  = max(0, int(len(majority) * target_ratio) - len(minority))
    extra    = [rng.choice(minority) for _ in range(n_extra)]
    balanced = majority + minority + extra
    rng.shuffle(balanced)
    n0 = sum(1 for _, l in balanced if l == 0)
    n1 = sum(1 for _, l in balanced if l == 1)
    print(f"[INFO] After oversample → Good:{n0}  Faulty:{n1}")
    return balanced


def _make_sampler(train_samples):
    labels  = [l for _, l in train_samples]
    n0, n1  = labels.count(0), labels.count(1)
    w0, w1  = 1.0 / max(n0, 1), 1.0 / max(n1, 1)
    weights = [w0 if l == 0 else w1 for l in labels]
    return WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.double),
        num_samples=len(weights), replacement=True)


def _make_loaders(train_samples, val_samples,
                   test_samples, args, heavy_aug=False):
    # Safe worker counts for Kaggle notebooks
    train_workers = min(max(2, args.workers), 4)
    val_workers   = min(train_workers, 4)

    # persistent_workers=True can deadlock on Kaggle with heavy augmentations;
    # only enable when workers > 0
    _persistent = train_workers > 0
    _common = dict(
        pin_memory         = True,
        persistent_workers = _persistent,
        timeout            = 120,
    )
    train_dl_kw = dict(**_common,
                       num_workers    = train_workers,
                       prefetch_factor= 4 if train_workers > 0 else 2)
    val_dl_kw   = dict(**_common,
                       num_workers    = val_workers,
                       prefetch_factor= 4 if val_workers > 0 else 2)

    if args.oversample:
        balanced        = _oversample_minority(train_samples, seed=args.seed)
        effective_train = balanced
    else:
        balanced        = train_samples
        effective_train = train_samples

    train_ds = RimDataset(
        balanced,
        _make_train_transform(args.img_size, heavy=heavy_aug),
        args.img_size,
        multi_scale=args.multi_scale,
        cache=False)

    train_loader = DataLoader(
        train_ds,
        batch_size  = args.batch,
        shuffle     = args.oversample,
        sampler     = None if args.oversample else _make_sampler(train_samples),
        drop_last   = True,
        **train_dl_kw)

    # Val: cached in RAM, single-pass batch size
    val_ds = RimDataset(
        val_samples,
        _make_val_transform(),
        args.img_size,
        multi_scale = False,
        cache       = args.cache_val)
    val_loader = DataLoader(
        val_ds,
        batch_size = min(args.batch * 2, 32),  # val: no grad, can use 2x batch
        shuffle    = False,
        **val_dl_kw)

    test_loader = None
    if test_samples:
        test_ds = RimDataset(
            test_samples,
            _make_val_transform(),
            args.img_size,
            multi_scale = False,
            cache       = args.cache_val)
        test_loader = DataLoader(
            test_ds,
            batch_size = args.batch,
            shuffle    = False,
            **val_dl_kw)

    mode = "oversampling" if args.oversample else "WeightedRandomSampler"
    print(f"[INFO] Imbalance strategy: {mode}")
    return train_loader, val_loader, test_loader, effective_train

When I started building this defect detection model, the biggest hurdle was that standard Convolutional Neural Networks (CNNs) treat every pixel with equal importance. That’s a massive problem when you’re inspecting industrial rims. A microscopic crack might only take up 1% of the image, while the other 99% is perfectly healthy, shiny metal. If the network pays equal attention to everything, the background noise drowns out the defect.

To fix this, I completely ripped out the standard pooling layers and wired in Attention Modules to teach the network to "see" more like a human inspector.

**1. The CBAM (Convolutional Block Attention Module)**

I injected a CBAM block right after the main feature-extraction backbones (like EfficientNet). Think of this as giving the network a pair of smart glasses. It operates in two sequential phases:
(Channel Attention): * What I did: I set up Max Pooling and Average Pooling to compress the spatial dimensions, feeding them into a shared multi-layer perceptron (MLP).
The backbone extracts hundreds of feature maps (channels). Some detect useful things like "scratches" or "edges," while others are distracted by "glare" or "background."
This channel attention step calculates which filters are actually firing at the defect and amplifies them, effectively muting the useless background channels.
(Spatial Attention): * What I did: I took those amplified channels, squashed them flat, and ran a $7 \times 7$ convolution over them to create a 2D map.
Why I did it: Now that the network knows what to look for, it needs to know where it is. This step creates a literal heatmap over the physical pixel coordinates (e.g., highlighting the bottom-left edge), forcing the network's mathematical focus exactly onto the physical location of the defect.

**2. AttentionPool2d (Replacing GAP):**

At the very end of the network, the industry standard is to use Global Average Pooling (GAP) to flatten the 2D feature maps into a 1D vector for the final classification layer. I completely threw GAP out. Instead, I wrote AttentionPool2d, which uses a $1 \times 1$ convolution paired with a Softmax function to assign a probability score to every single spatial coordinate.
GAP is terrible for anomaly detection. If you average a 1% crack with 99% healthy rim, the signal of the crack gets completely diluted and lost. By using a $1 \times 1$ convolution, my custom layer learns to score the importance of each region. Then, instead of a flat average, it takes a weighted sum. Mathematically, this allows the model to assign a weight of 0.0 to the healthy metal and 1.0 to the crack, funneling 100% of the network's decision-making power purely into the defective pixels.

**3. _get_out_channels (The Sanity Saver):** 

I wrote a tiny utility function that generates a dummy tensor of raw zeros, passes it through a given backbone, and returns the size of the output tensor.i did this because I built an ensemble of three completely different architectures (EfficientNet, RegNet, and ConvNeXt). EfficientNet might output 384 channels, while ConvNeXt outputs 768. Hardcoding these tensor dimensions is a nightmare and a recipe for massive shape-mismatch crashes if I ever decide to swap out a backbone later. Sending a dummy tensor through during initialization lets the code dynamically figure out its own matrix math sizes, keeping the pipeline modular and crash-proof.

The Strategy: Why use three backbones? (The "Wisdom of the Crowd")
When inspecting industrial rims, defects are rarely uniform. A defect could be a microscopic hairline crack, a deep gouge, or a subtle warping of the metal. If you rely on a single neural network architecture, it will inevitably have a "blind spot"—for example, it might be great at finding deep scratches but easily fooled by bright glare (false positives).
By using an ensemble of three fundamentally different architectural families, we create a "Wisdom of the Crowd" effect.
Diverse Perspectives: Because each backbone calculates math differently, they extract entirely different types of visual features from the exact same image.
The Cross-Attention Jury: Before making a final decision, the outputs of all three backbones are fed into a Multi-Head Cross-Attention layer. This acts like a jury room. If one backbone thinks it sees a crack, but the other two realize it's just a shadow based on their specific feature maps, they overrule the mistake. This drastically reduces false positives and creates a highly confident, robust model.
The Specialists: What each backbone doesTo make this work within Kaggle's strict 15GB VRAM limit, I selected three highly efficient, specialized "lightweight" models:
1. EfficientNet-B3 (The Generalist & Optimizer) It acts as the highly accurate generalist. EfficientNet is incredibly good at extracting multi-scale hierarchical features without wasting computational power. It looks at the overall global structure of the rim and quickly identifies areas where the standard geometry breaks down.
2. RegNetY-3.2GF (The Texture & Pattern Expert) Unlike models designed by human intuition, RegNets were designed by algorithms exploring massive "network design spaces." The 'Y' variant includes Squeeze-and-Excitation blocks that recalibrate channel weights dynamically.What it specializes in: Mathematical regularity. Because it is so highly regularized and structured, RegNetY excels at understanding the repetitive, machined textures of industrial metal.
3. ConvNeXt-Tiny (The Long-Range Tracker)What it specializes in: It uses unusually large convolution kernels ($7 \times 7$ grids, whereas older models use $3 \times 3$). Because it looks at much larger "patches" of the image at a single time, it is highly specialized at capturing long-range spatial dependencies. If there is a long, continuous stress fracture or a sweeping surface scratch that spans across a large portion of the rim, ConvNeXt will piece that continuous line together better than the other two.
Using exactly three backbones provides a natural "tie-breaker" if two models disagree, preventing the deadlocks that frequently occur when using only two. Conversely, adding a fourth backbone would immediately exceed Kaggle's 15GB VRAM limit while offering diminishing returns in accuracy.

In [7]:
class AttentionPool2d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Conv2d(channels, 1, kernel_size=1, bias=True)

    def forward(self, x):
        B, C, H, W = x.shape
        w = torch.softmax(
            self.score(x).view(B, 1, H * W), dim=2
        ).view(B, 1, H, W)
        return (x * w).sum(dim=(2, 3))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid          = max(channels // reduction, 8)
        self.ch_fc1  = nn.Linear(channels, mid, bias=False)
        self.ch_fc2  = nn.Linear(mid, channels, bias=False)
        self.sp_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        avg_c = F.adaptive_avg_pool2d(x, 1).flatten(1)
        max_c = F.adaptive_max_pool2d(x, 1).flatten(1)
        ch    = torch.sigmoid(
            self.ch_fc2(F.relu(self.ch_fc1(avg_c))) +
            self.ch_fc2(F.relu(self.ch_fc1(max_c)))
        ).unsqueeze(-1).unsqueeze(-1)
        x  = x * ch
        sp = torch.sigmoid(
            self.sp_conv(torch.cat(
                [x.mean(1, keepdim=True),
                 x.max(1, keepdim=True).values], dim=1)))
        return x * sp


def _get_out_channels(module, img_size=64, in_channels=3):
    device = next(module.parameters()).device
    with torch.no_grad():
        out = module(torch.zeros(1, in_channels,
                                  img_size, img_size,
                                  device=device))
    return out.shape[1]


class EnsembleModelV6(nn.Module):
    _arch_version = "v6_ensemble_kaggle"

    def __init__(self, use_grad_checkpoint=False, dropout_rate=0.35):
        super().__init__()
        self._use_grad_ckpt = use_grad_checkpoint

        # ── EfficientNet-B3 ───────────────────────────────────────────────
        eff          = models.efficientnet_b3(
            weights=models.EfficientNet_B3_Weights.DEFAULT)
        self.b4_stem = eff.features[:6]
        self.b4_last = eff.features[6:]
        _b4_ch       = _get_out_channels(nn.Sequential(self.b4_stem, self.b4_last))
        print(f"[INFO] EfficientNet-B3 : {_b4_ch}ch")
        self.b4_cbam = CBAM(_b4_ch)
        self.b4_pool = AttentionPool2d(_b4_ch)
        self.b4_proj = nn.Sequential(
            nn.Linear(_b4_ch, 320), nn.GELU(), nn.Dropout(dropout_rate))

        # ── RegNetY-8GF (stronger than ResNet-50) ─────────────────────────
        rn           = models.regnet_y_8gf(
            weights=models.RegNet_Y_8GF_Weights.DEFAULT)
        self.rn_stem = nn.Sequential(rn.stem, rn.trunk_output[:3])
        self.rn_last = rn.trunk_output[3:]
        _rn_ch       = _get_out_channels(nn.Sequential(self.rn_stem, self.rn_last))
        print(f"[INFO] RegNetY-8GF     : {_rn_ch}ch")
        self.rn_cbam = CBAM(_rn_ch)
        self.rn_pool = AttentionPool2d(_rn_ch)
        self.rn_proj = nn.Sequential(
            nn.Linear(_rn_ch, 320), nn.GELU(), nn.Dropout(dropout_rate))

        # ── ConvNeXt-Small ────────────────────────────────────────────────
        cx           = models.convnext_small(
            weights=models.ConvNeXt_Small_Weights.DEFAULT)
        self.cx_stem = cx.features[:6]
        self.cx_last = cx.features[6:]
        _cx_ch       = _get_out_channels(nn.Sequential(self.cx_stem, self.cx_last))
        print(f"[INFO] ConvNeXt-Small  : {_cx_ch}ch")
        self.cx_cbam = CBAM(_cx_ch)
        self.cx_pool = AttentionPool2d(_cx_ch)
        self.cx_proj = nn.Sequential(
            nn.Linear(_cx_ch, 320), nn.GELU(), nn.Dropout(dropout_rate))

        self.b4      = nn.ModuleList([self.b4_stem, self.b4_last,
                                       self.b4_cbam, self.b4_pool, self.b4_proj])
        self.resnet  = nn.ModuleList([self.rn_stem, self.rn_last,
                                       self.rn_cbam, self.rn_pool, self.rn_proj])
        self.convnxt = nn.ModuleList([self.cx_stem, self.cx_last,
                                       self.cx_cbam, self.cx_pool, self.cx_proj])

        # ── Fusion (320*3 = 960) ──────────────────────────────────────────
        fused_dim        = 320 * 3
        self.fusion_gate = nn.Sequential(
            nn.Linear(fused_dim, fused_dim), nn.Sigmoid())
        self.cross_attn  = nn.MultiheadAttention(
            embed_dim=320, num_heads=8, batch_first=True, dropout=0.1)
        self.cross_norm  = nn.LayerNorm(320)

        self.head = nn.Sequential(
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256),       nn.GELU(), nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, 128),       nn.GELU(), nn.Dropout(dropout_rate * 0.3))
        self.binary_head   = nn.Linear(128, 1)
        self.severity_head = nn.Linear(128, 1)

    def _fwd_backbone(self, stem, last, cbam, pool, proj, x):
        if self._use_grad_ckpt and self.training:
            feat = grad_checkpoint(stem, x,    use_reentrant=False)
            feat = grad_checkpoint(last, feat, use_reentrant=False)
        else:
            feat = last(stem(x))
        return proj(pool(cbam(feat)))

    def forward(self, x):
        f_b4 = self._fwd_backbone(
            self.b4_stem, self.b4_last, self.b4_cbam, self.b4_pool, self.b4_proj, x)
        f_rn = self._fwd_backbone(
            self.rn_stem, self.rn_last, self.rn_cbam, self.rn_pool, self.rn_proj, x)
        f_cx = self._fwd_backbone(
            self.cx_stem, self.cx_last, self.cx_cbam, self.cx_pool, self.cx_proj, x)

        tokens      = torch.stack([f_b4, f_rn, f_cx], dim=1)
        attn_out, _ = self.cross_attn(tokens, tokens, tokens)
        tokens      = self.cross_norm(tokens + attn_out)

        fused = torch.cat([tokens[:, 0], tokens[:, 1], tokens[:, 2]], dim=1)
        fused = fused * self.fusion_gate(fused)
        feat  = self.head(fused)
        return (self.binary_head(feat).squeeze(1),
                self.severity_head(feat).squeeze(1))


def build_model():
    return EnsembleModelV6(
        use_grad_checkpoint=args.grad_checkpoint,
        dropout_rate=args.dropout_rate)

**Loss Functions (The Grading System)**

We have a massive class imbalance. If 95% of the rims are perfect, the network can cheat and get 95% accuracy just by guessing "perfect" every single time.

I threw out standard Cross-Entropy loss and built a CombinedLoss function. I used Focal Loss to force the model to pay attention to "hard" examples (rare defects), and Asymmetric Loss (ASL) to aggressively discard the easy, boring "good" rims so they don't overwhelm the math. I also added a severity MSE as a bonus task to keep the network's gradients stable.

This forces the network to stop being lazy. It mathematically punishes the model for ignoring the rare anomalies.


**FocalLossWithLogits**
Standard Focal Loss for binary classification. Reduces the weight of
easy negative examples so the model focuses on hard positives (rare
defects). gamma=2.0 is a standard choice. Supports label smoothing
and class-weighted pos_weight.



**AsymmetricLoss**
ASL uses different focusing exponents for positive (gamma_pos=1) and
negative (gamma_neg=4) samples. It clips easy negatives more
aggressively than Focal Loss — better for class-imbalanced datasets.

**CombinedLoss**
Weighted combination:
total = (1 - 0.4) * FocalLoss + 0.4 * ASL + 0.25 * MSE(severity)
The severity MSE acts as a soft regulariser that shares gradient
with the binary head.


In [8]:
class FocalLossWithLogits(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, eps=0.0):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight
        self.eps        = eps

    def forward(self, logits, targets):
        if self.eps > 0:
            targets = targets * (1 - self.eps) + 0.5 * self.eps
        bce   = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets >= 0.5, probs, 1 - probs)
        return (((1 - pt) ** self.gamma) * bce).mean()


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps

    def forward(self, logits, targets):
        probs      = torch.sigmoid(logits)
        probs_neg  = (probs + self.clip).clamp(max=1.0)
        loss_pos   = targets       * torch.log(probs.clamp(min=self.eps))
        loss_neg   = (1 - targets) * torch.log((1 - probs_neg).clamp(min=self.eps))
        probs_pt   = probs * targets + (1 - probs) * (1 - targets)
        asl_weight = torch.where(
            targets == 1,
            (1 - probs_pt) ** self.gamma_pos,
            (1 - probs_pt) ** self.gamma_neg)
        return -((loss_pos + loss_neg) * asl_weight).mean()


class CombinedLoss(nn.Module):
    def __init__(self, focal_loss, asl_loss,
                 severity_weight=0.25, asl_weight=0.4):
        super().__init__()
        self.focal_loss      = focal_loss
        self.asl_loss        = asl_loss
        self.severity_weight = severity_weight
        self.asl_weight      = asl_weight

    def forward(self, bin_logits, sev_logits, targets):
        loss_focal = self.focal_loss(bin_logits, targets)
        loss_asl   = self.asl_loss(bin_logits,  targets)
        loss_bin   = (1 - self.asl_weight) * loss_focal + self.asl_weight * loss_asl
        loss_sev   = F.mse_loss(torch.sigmoid(sev_logits), targets)
        return loss_bin + self.severity_weight * loss_sev

****EMA (EXPONENTIAL MOVING AVERAGE)****


During training, a model's weights bounce around wildly. The very last step of training isn't necessarily the smartest version of the model—it might have just taken a bad step.

I implemented a ModelEMA (Exponential Moving Average). While the main model aggressively learns and bounces around, I keep a "shadow copy" in the background. After every step, this shadow model slowly absorbs a tiny fraction (0.9995 decay) of the main model's knowledge.

It creates a smoothed-out, highly stable version of the model. By averaging out the noisy gradient spikes, the shadow model almost always generalizes better to unseen data than the raw training model.

In [9]:
class ModelEMA:
    def __init__(self, model, decay=0.9998):
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)
        self.decay      = decay
        self.step_count = 0

    @torch.no_grad()
    def update(self, model):
        self.step_count += 1
        d = min(self.decay,
                (1 + self.step_count) / (10 + self.step_count))
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(d).add_(m.data, alpha=1 - d)
        for s, m in zip(self.shadow.buffers(), model.buffers()):
            s.copy_(m)


****MixUp / CutMix (The Hallucination Generator)****


I added a function that randomly sabotages the training batches.

MixUp takes two images and ghosts them over each other (like a double-exposure photograph).

CutMix literally cuts a square out of one image and glues it onto another.

I also blend their labels mathematically (e.g., telling the model "this image is now 70% perfect and 30% cracked").

Why: It prevents the model from memorizing specific images. By forcing it to look at Frankenstein-style hybrid rims, it learns to hunt for the actual physical textures of a defect, drastically improving its performance on real-world, unseen data.

In [10]:
def _mixup_cutmix(imgs, lbls, alpha=0.1):
    if alpha <= 0 or random.random() < 0.6:
        return imgs, lbls
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    if random.random() < 0.5:
        imgs = lam * imgs + (1 - lam) * imgs[idx]
        lbls = lam * lbls + (1 - lam) * lbls[idx]
    else:
        H, W  = imgs.shape[2:]
        cut_r = np.sqrt(1 - lam)
        cut_h = int(H * cut_r)
        cut_w = int(W * cut_r)
        cx = random.randint(0, W)
        cy = random.randint(0, H)
        x1 = max(0, cx - cut_w // 2)
        x2 = min(W, cx + cut_w // 2)
        y1 = max(0, cy - cut_h // 2)
        y2 = min(H, cy + cut_h // 2)
        imgs[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
        lam  = 1 - (x2 - x1) * (y2 - y1) / (H * W)
        lbls = lam * lbls + (1 - lam) * lbls[idx]
    return imgs, lbls

****TTA (Test-Time Augmentation): The "Second Opinion"****


The model might miss a subtle scratch just because of the specific angle the camera captured it.

Instead of asking the model for a prediction once, I flip and rotate the image 4 to 16 different ways, pass all of them through the network, and average the final probabilities.

why: It acts like a committee of inspectors looking at the same rim from multiple angles. It reduces the variance of the predictions and squeezes out the absolute maximum performance from the model without having to train it for a single extra minute.

In [11]:
def _tta_predict(model, imgs, device, n_views=4):
    """
    n_views=4  → flips only          (fast)
    n_views=16 → flips × rotations   (thorough, final eval only)
    """
    preds    = []
    base     = imgs.to(device)
    flip_fns = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.flip(x, [2]),
        lambda x: torch.flip(x, [2, 3]),
    ]
    if n_views <= 4:
        for fn in flip_fns[:n_views]:
            with torch.no_grad():
                logits, _ = model(fn(base))
            preds.append(torch.sigmoid(logits))
    else:
        for fn in flip_fns:
            for k in range(4):
                aug = torch.rot90(fn(base), k, [2, 3])
                with torch.no_grad():
                    logits, _ = model(aug)
                preds.append(torch.sigmoid(logits))
    return torch.stack(preds).mean(0)

****Validation & Threshold Sweep****

Standard classification assumes 50% (0.5) is the magic cutoff between "good" and "faulty." When your dataset is highly imbalanced (like the one in this case), 0.5 is almost always the wrong boundary.

I wrote a vectorized function that takes the raw validation probabilities and tests 90 different thresholds (from 0.05 to 0.96) simultaneously.

This mathematically guarantees we find the exact decision boundary that maximizes our F1 score. By doing this using broadcasting directly on the GPU, it calculates all 90 thresholds in a fraction of a second without a slow Python for loop.


In [12]:
@torch.no_grad()
def _validate(model, loader, device,
               threshold=0.5, use_tta=False,
               tta_views=4, profile=False):
    model.eval()
    all_probs, all_labels = [], []
    t_load = t_gpu = t_cpu = 0.0
    t0     = time.perf_counter()

    for imgs, lbls in tqdm(loader, desc="  val", leave=False):
        if profile:
            t_load += time.perf_counter() - t0

        t1 = time.perf_counter()
        if use_tta:
            probs = _tta_predict(model, imgs, device, n_views=tta_views)
        else:
            logits, _ = model(imgs.to(device))
            probs     = torch.sigmoid(logits)

        if profile and device.type == "cuda":
            torch.cuda.synchronize()
        if profile:
            t_gpu += time.perf_counter() - t1

        t2 = time.perf_counter()
        all_probs.append(probs.cpu())
        all_labels.append(lbls)
        if profile:
            t_cpu += time.perf_counter() - t2
            t0     = time.perf_counter()

    if profile:
        print(f"\n  [VAL PROFILE] "
              f"load={t_load:.2f}s  "
              f"gpu={t_gpu:.2f}s  "
              f"cpu_transfer={t_cpu:.2f}s")

    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds  = (probs >= threshold).astype(int)

    tp   = ((preds == 1) & (labels == 1)).sum()
    fp   = ((preds == 1) & (labels == 0)).sum()
    fn   = ((preds == 0) & (labels == 1)).sum()
    tn   = ((preds == 0) & (labels == 0)).sum()
    acc  = (tp + tn) / max(len(labels), 1)
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-8)
    auc  = (roc_auc_score(labels, probs)
            if _SKLEARN_OK and len(set(labels)) > 1 else None)
    return float(acc), float(f1), float(prec), float(rec), auc, probs, labels


def _sweep_threshold(model, loader, device,
                      title="", use_tta=False,
                      tta_views=4, profile=False):
    *_, probs, labels = _validate(
        model, loader, device,
        threshold=0.5, use_tta=use_tta,
        tta_views=tta_views, profile=profile)

    # Fully vectorised on GPU — no Python loop
    probs_t   = torch.tensor(probs,  device=device)
    labels_t  = torch.tensor(labels, device=device)
    thrs      = torch.arange(0.05, 0.96, 0.01, device=device)
    preds_all = (probs_t.unsqueeze(0) >= thrs.unsqueeze(1)).float()
    labels_2d = labels_t.unsqueeze(0)

    tp   = (preds_all * labels_2d).sum(1)
    fp   = (preds_all * (1 - labels_2d)).sum(1)
    fn   = ((1 - preds_all) * labels_2d).sum(1)
    prec = tp / (tp + fp).clamp(min=1)
    rec  = tp / (tp + fn).clamp(min=1)
    f1   = 2 * prec * rec / (prec + rec).clamp(min=1e-8)

    best_idx = f1.argmax()
    best_f1  = f1[best_idx].item()
    best_thr = thrs[best_idx].item()

    if title:
        print(f"[Threshold Sweep] {title}")
        print(f"  Best thr:{best_thr:.2f}  F1:{best_f1:.4f}")
    return best_thr, best_f1, probs, labels

****BN Recalibration: The "Reality Check"****

Our EMA "shadow" model (from the block 9) updates its weights via mathematical averaging, but it never actually has real images flowing through it during training. Because of this, its internal Batch Normalization (BN) running statistics (mean and variance) are completely stale.

Right before saving the best EMA model to disk, I force 25 batches of real training data through it in train mode (but without updating gradients).

Why: This forces the BN layers to recalculate their statistics based on reality. If I skipped this, the model would output garbage predictions at inference time.


In [13]:
def _recalibrate_bn(model, loader, device, n_batches=25):
    print(f"  [BN recal] {n_batches} batches…", flush=True)
    model.train()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(
                tqdm(loader, total=n_batches,
                     desc="  BN recal", leave=False)):
            if i >= n_batches:
                break
            model(imgs.to(device))
    model.eval()
    print("  [BN recal] done.", flush=True)

****Platt Scaling: The "Confidence Fixer"****

Neural networks are notoriously arrogant. When they output a 99% probability, they are frequently wrong. They are uncalibrated.

I fit a simple Logistic Regression curve over the raw outputs (logits) using the validation set to create a .pkl calibrator file.

This aligns the model's confidence with reality. If the calibrated output says there is an 80% chance of a defect, it means 8 out of 10 times, there actually is one. It makes our threshold sweeps much more trustworthy.


In [14]:
def _calibrate_model(model, val_loader, device,
                      cached_probs=None, cached_labels=None):
    if not _SKLEARN_OK:
        print("[WARN] sklearn missing — skip calibration.")
        return None
    print("[INFO] Fitting Platt scaling…")
    if cached_probs is None or cached_labels is None:
        *_, probs, labels = _validate(model, val_loader, device)
    else:
        probs, labels = cached_probs, cached_labels
    cal = LogisticRegression(C=1.0, max_iter=1000)
    cal.fit(probs.reshape(-1, 1), labels.astype(int))
    out_path = Path(args.out).parent / "calibrator.pkl"
    with open(out_path, "wb") as f:
        pickle.dump(cal, f)
    print(f"[INFO] Calibrator → {out_path}")
    return cal

****Three-Phase Training: "Progressive Unfreezing"****


The backbones are pre-trained on millions of generic images. If I blast them with a high learning rate and a totally random classification head on Step 1, the massive gradient updates will permanently destroy all the useful edge-detecting features the backbones already learned.


*Phase 1:* Freeze the entire backbone. Train only the new classification head.


*Phase 2:* Unfreeze just the final blocks of the backbones using a tiny learning rate.


*Phase 3:* Unfreeze everything and hit it with heavy augmentations.

Why: This is a transfer learning guardrail. It gently adapts the network to our specific industrial rims without causing catastrophic forgetting of its pre-trained "smarts."

In [15]:
def _setup_phase(model, phase, args, n_epochs):
    base_lr  = args.lr
    is_cuda  = next(model.parameters()).device.type == "cuda"

    backbone_ids = set()
    for mod_list in [model.b4, model.resnet, model.convnxt]:
        for m in mod_list:
            backbone_ids.update(id(p) for p in m.parameters())
    head_params = [p for p in model.parameters()
                   if id(p) not in backbone_ids]

    sep = "─" * 70
    print(f"\n{sep}")

    if phase == 1:
        print(f"[Phase 1] Head warm-up | {n_epochs} epochs")
        for p in model.parameters():
            p.requires_grad = False
        for p in head_params:
            p.requires_grad = True
        for p in model.cross_attn.parameters():
            p.requires_grad = True
        for p in model.fusion_gate.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=base_lr, weight_decay=1e-4)

    elif phase == 2:
        print(f"[Phase 2] Head + last blocks | {n_epochs} epochs")
        for p in model.parameters():
            p.requires_grad = False
        for p in head_params:
            p.requires_grad = True
        # rn_last for RegNetY is trunk_output[3] — same attribute name, works fine
        for mod in [model.rn_last, model.b4_last, model.cx_last]:
            for p in mod.parameters():
                p.requires_grad = True
        optimizer = torch.optim.AdamW([
            {"params": [p for p in head_params if p.requires_grad],
             "lr": base_lr},
            {"params": list(model.rn_last.parameters()), "lr": base_lr * 0.1},
            {"params": list(model.b4_last.parameters()), "lr": base_lr * 0.1},
            {"params": list(model.cx_last.parameters()), "lr": base_lr * 0.1},
        ], weight_decay=1e-4, fused=is_cuda)

    elif phase == 3:
        print(f"[Phase 3] Full model | {n_epochs} epochs")
        for p in model.parameters():
            p.requires_grad = True

        def _collect(ml):
            return [p for m in ml for p in m.parameters()]

        optimizer = torch.optim.AdamW([
            {"params": _collect(model.b4),      "lr": base_lr * 0.15},
            {"params": _collect(model.resnet),  "lr": base_lr * 0.15},
            {"params": _collect(model.convnxt), "lr": base_lr * 0.10},
            {"params": head_params,             "lr": base_lr},
        ], weight_decay=5e-5, fused=is_cuda)
    else:
        raise ValueError(f"Unknown phase: {phase}")

    n_train = sum(p.numel() for p in model.parameters()
                  if p.requires_grad) / 1e6
    print(f"  Trainable: {n_train:.2f}M")
    print(sep)
    return optimizer

****AMP Setup****


Automatic Mixed Precision (fp16) makes training much faster and uses less VRAM. However, with massive ensemble models, fp16 math is too imprecise and often causes gradients to explode into "NaN" (Not a Number), instantly ruining the training run.

to fix this i wrote a hardware detector. If the code is running on a modern GPU (like an A100 or L4), it turns on bfloat16—a much more stable precision format. If it detects an older GPU (like Kaggle's T4), it completely disables AMP and falls back to safe fp32.

Why: It guarantees the code won't crash halfway through a 10-hour training run just because of a hardware quirk.

In [16]:
def _setup_amp(device, want_bf16):
    """
    Returns (use_amp, amp_dtype, scaler).

    Rules:
      - Ampere+ (sm_80+, e.g. A100/L4): bf16 AMP, no scaler needed
      - Volta/Turing (sm_70/75, e.g. V100/T4): fp16 + GradScaler (safe with grad_clip)
      - CPU: no AMP
    """
    if device.type != "cuda":
        print("[INFO] AMP: disabled (CPU)")
        return False, torch.float32, None

    major = torch.cuda.get_device_capability(device)[0]
    name  = torch.cuda.get_device_name(device)

    if want_bf16 and major >= 8:
        print(f"[INFO] AMP: bf16 (Ampere+ GPU: {name})")
        return True, torch.bfloat16, None   # bf16 stable, no scaler needed
    elif major >= 7:
        # T4 (sm_75), V100 (sm_70): fp16 + GradScaler is safe
        print(f"[INFO] AMP: fp16 + GradScaler ({name})")
        scaler = torch.amp.GradScaler("cuda")
        return True, torch.float16, scaler
    else:
        print(f"[INFO] AMP: disabled on {name} (sm_{major*10})")
        return False, torch.float32, None


****The Main Training Loop (The "Classroom Semester")****



If the neural network is a student, the Main Training Loop is the semester schedule. It dictates how the student studies, how they are tested, and how we prevent them from burning out. Here is the step-by-step logic I built to manage this grueling process:

**1. Setting the Rules (Seed & Device Setup)**

Computers are inherently chaotic. If we run an experiment on Monday and get 95% accuracy, we need to be able to hit "Run" on Tuesday and get the exact same result to prove our code actually works.

I locked down the "Random Seeds." I also put the model on the fastest piece of hardware available (the GPU) and turned on optimizations like TF32 (TensorFloat-32), which acts like a math shortcut that allows the GPU to calculate matrix multiplications significantly faster without losing accuracy.

**2. Building the Brain (Model & SWA / Resume Setup)**

I instantiate the 3-backbone ensemble. I also added a torch.compile flag and a "Channels-Last" memory layout. Finally, I check if a resume.pt file exists on the hard drive.

The memory layout and compiling are purely for speed—it reorganizes how the data sits in the microchips so the GPU can read it 20% faster. The resume setup is disaster prevention. Training takes hours. If Kaggle crashes or the power goes out at Epoch 20, the model loads the checkpoint and picks up exactly where it left off, rather than starting over from kindergarten.

**3. The Epoch Loop (The Daily Grind)**
An "Epoch" means the student has looked at every single image in the textbook exactly one time.

The "Progressive Unfreezing" Transitions: We don't teach calculus on day one. During Phase 1, I literally lock the student's core brain (accumulation disabled, backbone frozen) and only let them learn the absolute basics (training the head). At Phase 3, the training wheels come off—everything is unfrozen, and we hit them with heavy, confusing image augmentations to make them an expert.

The Batch Loop (Studying page by page):

MixUp/CutMix: I hand the model a deliberately confusing, blended image.

Forward Pass: The model looks at the image and takes a guess ("I am 80% sure this rim is faulty").

Loss: The code acts as the harsh grader. ("Wrong, this rim is perfectly fine. Your error is huge.")

Backward Pass & Optimizer Step: The code reaches back into the model's brain and tweaks the microscopic math weights so it won't make that same mistake next time.

The "NaN Guard" (Preventing Brain Aneurysms): Sometimes, during the backward pass, the math literally explodes to infinity (called a NaN, or Not-a-Number). If a single NaN enters the model's brain, it corrupts the entire network instantly, ruining hours of training.

I wrote a scanner. If it detects infinite math, it deletes that batch, skips the update, prints a warning, and moves on safely.

The "VRAM Guard" (Clearing the Desk):

A GPU has limited short-term memory. Passing images through the network fills this memory like stacking textbooks on a desk. Eventually, the desk collapses (an Out-Of-Memory crash).

I wrote a trigger. If the desk gets too full (over 13.5GB), the script calls torch.cuda.empty_cache(), which acts like a robotic arm sweeping all the unneeded textbooks off the desk so we can keep studying without a crash.

**4. The Midterms (Validation & Early Stopping)**

After every few epochs, the studying stops. We bring out the Validation Set (images the model has never seen). We test the model and find its best threshold.

**5.Post-Training Evaluation (The Final Audit)**
The semester is over, but we have multiple versions of the model floating around (the last epoch, the SWA averaged model, the best checkpoint). Which one actually goes into production?

I wrote an automated audit. It loads up the absolute best checkpoint, calibrates its confidence using Platt Scaling, and then forces it to take a final, massive 16-view Test-Time Augmentation (TTA) exam.

In [17]:
def train():
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.deterministic = False

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        print(f"[INFO] GPU: {torch.cuda.get_device_name(device)}")
        torch.backends.cudnn.benchmark        = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32       = True
    else:
        print("[INFO] CPU mode")

    use_amp, amp_dtype, scaler = _setup_amp(device, args.bf16)

    # ── Data ──────────────────────────────────────────────────────────────
    all_samples = _load_samples(args.dataset, args.csv)
    train_samples, val_samples, test_samples = _split_samples(
        all_samples, args.val_split, args.test_split, seed=args.seed)

    assets = Path(args.out).parent
    assets.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(val_samples, columns=["image_id", "target"]).to_csv(
        assets / "val_split.csv", index=False)

    # Light aug for phases 1 & 2
    (train_loader, val_loader,
     test_loader, effective_train) = _make_loaders(
        train_samples, val_samples, test_samples,
        args, heavy_aug=False)

    # ── Model ─────────────────────────────────────────────────────────────
    print("\n[INFO] Building model…")
    model = build_model().to(device)
    if args.channels_last and device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
        print("[INFO] channels-last layout")

    total_p = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"[INFO] Total parameters: {total_p:.1f}M")

    if args.compile and device.type == "cuda" and _DYNAMO_OK:
        try:
            torch._dynamo.config.suppress_errors = True
            model = torch.compile(model, mode="max-autotune")
            print("[INFO] torch.compile active")
        except Exception as e:
            print(f"[WARN] torch.compile failed: {e}")

    ema = ModelEMA(model, decay=args.ema_decay) if args.ema_decay > 0 else None

    # Loss
    n0    = sum(1 for _, l in effective_train if l == 0)
    n1    = sum(1 for _, l in effective_train if l == 1)
    pos_w = torch.tensor([n0 / max(n1, 1)],
                          dtype=torch.float32).to(device)
    focal     = FocalLossWithLogits(
        gamma=args.focal_gamma, pos_weight=pos_w, eps=args.label_smooth)
    asl       = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
    criterion = CombinedLoss(focal, asl, severity_weight=0.25, asl_weight=0.4)
    print(f"[INFO] pos_weight={pos_w.item():.3f}")

    # SWA
    swa_model     = None
    swa_scheduler = None
    if args.swa_start > 0:
        try:
            swa_model = AveragedModel(model)
            print(f"[INFO] SWA starts at epoch {args.swa_start}")
        except Exception as e:
            print(f"[WARN] SWA init failed: {e}")

    best_val_f1       = -1.0
    no_improve_epochs = 0
    optimizer         = None
    scheduler         = None
    current_phase     = 0
    start_epoch       = 1
    phase1_end        = args.phase1_epochs
    phase2_end        = args.phase1_epochs + args.phase2_epochs
    nan_count         = 0

    # ── Resume ────────────────────────────────────────────────────────────
    if args.resume and Path(args.resume).exists():
        print(f"[INFO] Resuming from {args.resume}")
        ckpt        = torch.load(args.resume, map_location=device,
                                  weights_only=False)
        model.load_state_dict(ckpt["state_dict"], strict=False)
        start_epoch = int(ckpt.get("epoch", 1)) + 1
        best_val_f1 = float(ckpt.get("best_val_f1", -1.0))
        if ema and "ema_state" in ckpt:
            ema.shadow.load_state_dict(ckpt["ema_state"], strict=False)
        print(f"  → start epoch {start_epoch}  best F1={best_val_f1:.4f}")
    else:
        print("[INFO] Training from scratch")

    print(f"\n{'═'*60}")
    print(f"  Training {args.epochs} epochs  |  "
          f"batch={args.batch}  accum={args.accum_steps}  "
          f"workers={args.workers}  amp={use_amp}")
    print(f"{'═'*60}\n")

    for epoch in range(start_epoch, args.epochs + 1):

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        # ── Phase transitions ─────────────────────────────────────────────
        if epoch <= phase1_end and current_phase != 1:
            current_phase = 1
            optimizer = _setup_phase(model, 1, args, args.phase1_epochs)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=args.cos_restart_period, T_mult=1, eta_min=1e-7)

        elif phase1_end < epoch <= phase2_end and current_phase != 2:
            current_phase = 2
            optimizer = _setup_phase(model, 2, args, args.phase2_epochs)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=args.cos_restart_period, T_mult=1, eta_min=1e-7)

        elif epoch > phase2_end and current_phase != 3:
            current_phase = 3
            print("[INFO] Switching to heavy augmentation for phase 3…")
            (train_loader, val_loader,
             test_loader, effective_train) = _make_loaders(
                train_samples, val_samples, test_samples,
                args, heavy_aug=True)
            optimizer = _setup_phase(model, 3, args, args.phase3_epochs)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=args.phase3_epochs * len(train_loader), eta_min=1e-7)

        # ── Train one epoch ───────────────────────────────────────────────
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        n_steps      = 0
        nan_skipped  = 0

        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch:02d}/{args.epochs}",
                    leave=True)

        for step, (imgs, lbls) in enumerate(pbar):
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)

            if args.channels_last and device.type == "cuda":
                imgs = imgs.to(memory_format=torch.channels_last)

            imgs, lbls = _mixup_cutmix(imgs, lbls, alpha=args.mixup_alpha)

            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                bin_logits, sev_logits = model(imgs)
                loss = criterion(bin_logits, sev_logits, lbls) / args.accum_steps

            # ── NaN guard ─────────────────────────────────────────────────
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skipped += 1
                nan_count   += 1
                optimizer.zero_grad(set_to_none=True)
                if nan_count % 20 == 0:
                    print(f"\n[WARN] {nan_count} NaN/Inf losses total — "
                          f"check your data / lr.")
                continue
            # ─────────────────────────────────────────────────────────────

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            running_loss += loss.item() * args.accum_steps
            n_steps      += 1

            if (step + 1) % args.accum_steps == 0:
                if scaler is not None:
                    if args.grad_clip > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(
                            [p for p in model.parameters() if p.requires_grad],
                            args.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    if args.grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(
                            [p for p in model.parameters() if p.requires_grad],
                            args.grad_clip)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if ema:
                    ema.update(model)
                if scheduler:
                    scheduler.step(epoch - 1 + step / len(train_loader))

            pbar.set_postfix(
                loss=f"{loss.item() * args.accum_steps:.4f}",
                phase=current_phase,
                nan=nan_skipped,
                lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        if nan_skipped > 0:
            print(f"  [WARN] {nan_skipped} batches skipped (NaN/Inf) this epoch.")

        # SWA update
        if swa_model is not None and epoch >= args.swa_start:
            swa_model.update_parameters(model)
            if swa_scheduler is None:
                swa_scheduler = torch.optim.swa_utils.SWALR(
                    optimizer, swa_lr=args.lr * 0.1, anneal_epochs=5)
            swa_scheduler.step()

        # ── Skip validation on off-cycles ─────────────────────────────────
        if args.val_freq > 1 and epoch % args.val_freq != 0 \
                and epoch != args.epochs:
            avg_loss = running_loss / max(n_steps, 1)
            print(f"\nEpoch {epoch:02d} train_loss={avg_loss:.4f}"
                  f" — val skipped (val_freq={args.val_freq})",
                  flush=True)
            continue

        # ── Validation ────────────────────────────────────────────────────
        eval_model = ema.shadow if ema else model

        # TTA only at specified frequency; default freq=0 means never during training
        use_tta_now = (args.tta_freq > 0 and epoch % args.tta_freq == 0)

        # Profile first epoch only
        profile_now = (args.profile_val and epoch == start_epoch)

        avg_loss = running_loss / max(n_steps, 1)
        tta_str  = f" [TTA {args.tta_views_final}-view]" if use_tta_now else ""
        print(f"\nEpoch {epoch:02d} train_loss={avg_loss:.4f}"
              f" — validating{tta_str}…", flush=True)

        val_t0 = time.perf_counter()
        swept_thresh, swept_f1, probs, labels = _sweep_threshold(
            eval_model, val_loader, device,
            use_tta   = use_tta_now,
            tta_views = args.tta_views_final,
            profile   = profile_now)
        val_secs = time.perf_counter() - val_t0

        preds    = (probs >= swept_thresh).astype(int)
        tp = ((preds == 1) & (labels == 1)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()

        val_prec = tp / max(tp + fp, 1)
        val_rec  = tp / max(tp + fn, 1)
        val_auc  = (roc_auc_score(labels, probs)
                    if _SKLEARN_OK and len(set(labels)) > 1 else None)

        auc_str = f"  AUC:{val_auc:.4f}" if val_auc else ""
        print(f"  F1:{swept_f1:.4f}"
              f"  Prec:{val_prec:.4f}"
              f"  Rec:{val_rec:.4f}"
              f"{auc_str}"
              f"  thr:{swept_thresh:.2f}"
              f"  val_time:{val_secs:.1f}s"
              f"{tta_str}", flush=True)

        # ── Save best ─────────────────────────────────────────────────────
        if swept_f1 > best_val_f1:
            best_val_f1       = swept_f1
            no_improve_epochs = 0

            if ema:
                _recalibrate_bn(ema.shadow, train_loader, device,
                                 args.bn_recal_batches)
                save_state = ema.shadow.state_dict()
            else:
                save_state = model.state_dict()

            ckpt = {
                "state_dict":     save_state,
                "arch_version":   EnsembleModelV6._arch_version,
                "epoch":          int(epoch),
                "best_val_f1":    float(best_val_f1),
                "best_threshold": float(swept_thresh),
                "ema_state":      ema.shadow.state_dict() if ema else None,
            }
            Path(args.out).parent.mkdir(parents=True, exist_ok=True)
            torch.save(ckpt, args.out)
            print(f"  ✓ Saved  F1={best_val_f1:.4f}"
                  f"  thr={swept_thresh:.2f}"
                  f"  → {args.out}", flush=True)
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= args.early_stop:
                print(f"\n[INFO] Early stop — "
                      f"{args.early_stop} epochs without improvement.")
                break
            elif no_improve_epochs >= args.early_stop - 4:
                print(f"  [WARN] No improvement for {no_improve_epochs} epoch(s).")

    # ── SWA finalisation ──────────────────────────────────────────────────
    if swa_model is not None:
        print("\n[INFO] SWA: recalculating BN…")
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        swa_path = str(args.out).replace(".pt", "_swa.pt")
        torch.save({
            "state_dict":   swa_model.module.state_dict(),
            "arch_version": "swa_" + EnsembleModelV6._arch_version,
        }, swa_path)
        print(f"[INFO] SWA checkpoint → {swa_path}")
        swa_eval = build_model().to(device)
        swa_eval.load_state_dict(swa_model.module.state_dict())
        _sweep_threshold(swa_eval, val_loader, device,
                          title="SWA Final", use_tta=True,
                          tta_views=args.tta_views_final)

    print(f"\n{'═'*60}")
    print(f"  DONE  |  Best val F1: {best_val_f1:.4f}")
    print(f"{'═'*60}")

    # ── Post-training ─────────────────────────────────────────────────────
    print("[INFO] Loading best checkpoint…")
    best_model = build_model().to(device)
    if args.channels_last and device.type == "cuda":
        best_model = best_model.to(memory_format=torch.channels_last)
    ckpt = torch.load(args.out, map_location=device, weights_only=False)
    best_model.load_state_dict(ckpt["state_dict"])

    # Single pass — reused for calibration (no extra val pass)
    print("[INFO] Final evaluation (no TTA)…")
    best_thr, best_f1, probs, labels = _sweep_threshold(
        best_model, val_loader, device,
        title="Final — no TTA", use_tta=False)

    if args.calibrate:
        _calibrate_model(best_model, val_loader, device,
                          cached_probs=probs, cached_labels=labels)

    # Full TTA only once at the very end
    _sweep_threshold(best_model, val_loader, device,
                      title="Final — 16-view TTA",
                      use_tta=True, tta_views=args.tta_views_final)

    if test_loader:
        _, test_f1, _, _, test_auc, *_ = _validate(
            best_model, test_loader, device,
            use_tta=True, tta_views=args.tta_views_final)
        print(f"[INFO] Test  F1={test_f1:.4f}"
              + (f"  AUC={test_auc:.4f}" if test_auc else ""))

****Main****

just calling the training function


In [18]:
if __name__ == "__main__":
    train()

[INFO] GPU: Tesla T4
[INFO] AMP: fp16 + GradScaler (Tesla T4)
[INFO] Loaded 35342 samples.
[INFO] Split → Train:31102  Val:4240  Test:0
[INFO] Balance → Good:12962  Faulty:18140  Ratio:0.71:1
[INFO] After oversample → Good:18140  Faulty:18140
  [Cache] Loading 4240 images into RAM…


/tmp/ipykernel_23/2264116826.py:15: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
                                                              

  [Cache] Done.


[INFO] Imbalance strategy: oversampling

[INFO] Building model…
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 142MB/s]


...

  F1:0.9341  Prec:0.9207  Rec:0.9478  AUC:0.9748  thr:0.42  val_time:107.0s



[INFO] SWA: recalculating BN…
[INFO] SWA checkpoint → /kaggle/working/assets/model_v6_swa.pt
[INFO] EfficientNet-B3 : 1536ch
[INFO] RegNetY-8GF     : 2016ch
[INFO] ConvNeXt-Small  : 768ch


[Threshold Sweep] SWA Final
  Best thr:0.41  F1:0.9339

════════════════════════════════════════════════════════════
  DONE  |  Best val F1: 0.9341
════════════════════════════════════════════════════════════
[INFO] Loading best checkpoint…
[INFO] EfficientNet-B3 : 1536ch
[INFO] RegNetY-8GF     : 2016ch
[INFO] ConvNeXt-Small  : 768ch
[INFO] Final evaluation (no TTA)…


[Threshold Sweep] Final — no TTA
  Best thr:0.42  F1:0.9342
[INFO] Fitting Platt scaling…
[INFO] Calibrator → /kaggle/working/assets/calibrator.pkl


[Threshold Sweep] Final — 16-view TTA
  Best thr:0.44  F1:0.9337


****EVALUATION & METRICS REPORT****

This section acts as a deep dive into the model's performance on the validation split. While the training loop gives us quick metrics, this block provides a comprehensive audit:

**1. Probability Distribution:** It shows how confident the model is. 

**2. Threshold Sweep:** It tests 100 different probability cutoffs (from 0.01 to 0.99) to find the absolute best F1 score decision boundary.

**3. Error Analysis:** It isolates and lists the exact files the model misclassified, separating them into False Positives (Good rims marked as Faulty) and False Negatives (Faulty rims missed by the model). This is crucial for debugging the model's blind spots.

In [19]:

COMPARE_VAL_CSV   = f"{Path(args.out).parent}/val_split.csv"
COMPARE_MODEL     = args.out
COMPARE_THRESHOLD = 0.50
COMPARE_TTA       = True
COMPARE_BATCH     = 24

def compute_metrics(y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    n      = len(y_true)
    acc    = (tp + tn) / n if n > 0 else 0.0
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = (2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0)
    return dict(acc=acc, prec=prec, recall=recall, f1=f1, tp=tp, tn=tn, fp=fp, fn=fn)

def run_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'═'*60}\n  EnsembleModelV6 — Evaluation Audit\n{'═'*60}")
    
    # Load Validation Samples
    df = pd.read_csv(COMPARE_VAL_CSV)
    val_samples = [(row["image_id"], int(row["target"])) for _, row in df.iterrows()]
    
    dataset = RimDataset(val_samples, _make_val_transform(), args.img_size, cache=False)
    loader  = DataLoader(dataset, batch_size=COMPARE_BATCH, shuffle=False, num_workers=2)

    # Load Model
    model = build_model().to(device)
    ckpt = torch.load(COMPARE_MODEL, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    
    bt = ckpt.get("best_threshold", COMPARE_THRESHOLD)
    print(f"[INFO] Loaded checkpoint. Training Best Threshold: {bt:.2f}\n")

    all_probs, all_labels, all_paths = [], [], []

    # Inference Loop
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs = imgs.to(device, non_blocking=True)
            if COMPARE_TTA:
                probs = _tta_predict(model, imgs, device, n_views=16).cpu().numpy()
            else:
                logits, _ = model(imgs)
                probs = torch.sigmoid(logits).cpu().numpy()
                
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.numpy().astype(int).tolist())

    arr = np.array(all_probs)
    print(f"\n[Probability Distribution]")
    print(f"  Min/Max  : {arr.min():.4f} / {arr.max():.4f}")
    print(f"  Mean     : {arr.mean():.4f}")
    
    # Threshold Sweep
    best_f1, best_thresh = -1.0, COMPARE_THRESHOLD
    for t_int in range(1, 99):
        t = t_int / 100.0
        preds = [1 if p >= t else 0 for p in all_probs]
        m = compute_metrics(all_labels, preds)
        if m["f1"] > best_f1:
            best_f1, best_thresh = m["f1"], t

    print(f"\n[Threshold Sweep - Snapshot]")
    for t_int in range(10, 96, 10):
        t = t_int / 100.0
        preds = [1 if p >= t else 0 for p in all_probs]
        m = compute_metrics(all_labels, preds)
        mark = f"  ← best≈{best_thresh:.2f}" if abs(t - best_thresh) < 0.05 else ""
        print(f"  Thr {t:>4.2f} | F1: {m['f1']:>6.4f} | Prec: {m['prec']:>6.4f} | Rec: {m['recall']:>6.4f} {mark}")

    # Final Report & Misclassifications
    final_preds = [1 if p >= best_thresh else 0 for p in all_probs]
    m = compute_metrics(all_labels, final_preds)
    
    print(f"\n{'═'*58}\n  FINAL REPORT (Threshold = {best_thresh:.2f})\n{'═'*58}")
    print(f"  Accuracy: {m['acc']:.2%} | F1: {m['f1']:.4f} | Prec: {m['prec']:.2%} | Rec: {m['recall']:.2%}")
    print(f"  TP: {m['tp']:>4} (Correct Faulty)   |  TN: {m['tn']:>4} (Correct Good)")
    print(f"  FP: {m['fp']:>4} (False Alarms)     |  FN: {m['fn']:>4} (Missed Defects)")
    
    # Error listing
    paths_list = [s[0] for s in val_samples]
    errors = [(p, l, pr, prob) for p, l, pr, prob in zip(paths_list, all_labels, final_preds, all_probs) if l != pr]
    errors.sort(key=lambda x: abs(x[3] - 0.5), reverse=True)
    
    if errors:
        print(f"\n[Misclassified Images — Worst {min(10, len(errors))} predictions]")
        for path, l, pr, prob in errors[:10]:
            print(f"  {Path(path).name:<30} | True: {l} | Pred: {pr} | Prob: {prob:.4f}")
            
if __name__ == "__main__":
    run_evaluation()


════════════════════════════════════════════════════════════
  EnsembleModelV6 — Evaluation Audit
════════════════════════════════════════════════════════════
[INFO] EfficientNet-B3 : 1536ch
[INFO] RegNetY-8GF     : 2016ch
[INFO] ConvNeXt-Small  : 768ch
[INFO] Loaded checkpoint. Training Best Threshold: 0.43



Evaluating: 100%|██████████| 177/177 [28:55<00:00,  9.81s/it]



[Probability Distribution]
  Min/Max  : 0.0711 / 0.9994
  Mean     : 0.5964

[Threshold Sweep - Snapshot]
  Thr 0.10 | F1: 0.7413 | Prec: 0.5891 | Rec: 0.9996 
  Thr 0.20 | F1: 0.8697 | Prec: 0.7760 | Rec: 0.9891 
  Thr 0.30 | F1: 0.9204 | Prec: 0.8755 | Rec: 0.9701 
  Thr 0.40 | F1: 0.9318 | Prec: 0.9111 | Rec: 0.9535   ← best≈0.44
  Thr 0.50 | F1: 0.9300 | Prec: 0.9383 | Rec: 0.9220 
  Thr 0.60 | F1: 0.9193 | Prec: 0.9628 | Rec: 0.8795 
  Thr 0.70 | F1: 0.8992 | Prec: 0.9818 | Rec: 0.8294 
  Thr 0.80 | F1: 0.8597 | Prec: 0.9926 | Rec: 0.7582 
  Thr 0.90 | F1: 0.7647 | Prec: 0.9968 | Rec: 0.6203 

══════════════════════════════════════════════════════════
  FINAL REPORT (Threshold = 0.44)
══════════════════════════════════════════════════════════
  Accuracy: 92.17% | F1: 0.9337 | Prec: 92.30% | Rec: 94.46%
  TP: 2336 (Correct Faulty)   |  TN: 1572 (Correct Good)
  FP:  195 (False Alarms)     |  FN:  137 (Missed Defects)

[Misclassified Images — Worst 10 predictions]
  0a3c784a-21e6-4

****INFERENCE & SUBMISSION (predict.py)****

This pipeline takes completely unlabelled, raw test images and generates the final `submission.csv` for Kaggle scoring. 

**What it does:**
1. **Dynamic Cropping:** It runs the `extract_rim_crop` function using OpenCV's HoughCircles. It detects the physical metal rim in the raw test image and crops out the useless background, standardising it to match the training data.

2. **Test-Time Augmentation (TTA):** It applies flip and rotation augmentations to every test image, forcing the model to analyze the rim from 6 to 16 different perspectives before voting on the final probability.

3. **Thresholding & Output:** It applies the optimized threshold (found during the validation sweep) to convert the raw probabilities into binary 0 or 1 labels, finally exporting a Pandas DataFrame to a CSV file.

In [20]:

TEST_DIR       = "/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images"
SUBMISSION_OUT = "/kaggle/working/submission.csv"
PREDICT_TTA    = True

def extract_rim_crop(img_bgr: np.ndarray, size: int = 260, wide_scale: float = 1.0) -> np.ndarray:
    """Uses OpenCV HoughCircles to find the rim and crop the background."""
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT,
        dp=1.2, minDist=200, param1=60, param2=35,
        minRadius=100, maxRadius=600,
    )
    if circles is not None:
        cx, cy, r = np.round(circles[0, 0]).astype(int)
        rim_w  = max(20, int(r * 0.18))
        margin = int((rim_w + 10) * wide_scale)
        h, w   = gray.shape
        x1 = max(0, cx - r - margin); y1 = max(0, cy - r - margin)
        x2 = min(w, cx + r + margin); y2 = min(h, cy + r + margin)
        crop = img_bgr[y1:y2, x1:x2]
    else:
        crop = img_bgr
    return cv2.cvtColor(cv2.resize(crop, (size, size)), cv2.COLOR_BGR2RGB)

class UnlabelledTestDataset(Dataset):
    def __init__(self, image_paths, transform, img_size):
        self.paths = image_paths
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read: {path}")
        crop = extract_rim_crop(img_bgr, self.img_size)
        
        if _ALBU_OK:
            tensor = self.transform(image=crop)["image"]
        else:
            tensor = self.transform(Image.fromarray(crop))
            
        return tensor, path

def generate_submission():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[INFO] Starting Inference pipeline on {device}")
    
    # Verify test images exist
    paths = sorted([str(p) for p in Path(TEST_DIR).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    if not paths:
        print(f"[WARN] No images found in {TEST_DIR}. Ensure path is correct.")
        return
        
    print(f"[INFO] Found {len(paths)} test images.")

    # Load Model
    model = build_model().to(device)
    ckpt = torch.load(args.out, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    
    # Best threshold from training/compare phase
    best_thresh = ckpt.get("best_threshold", 0.5)

    dataset = UnlabelledTestDataset(paths, _make_val_transform(), args.img_size)
    loader  = DataLoader(dataset, batch_size=args.batch, shuffle=False, num_workers=2)

    image_ids, all_probs = [], []

    # Predict loop
    with torch.no_grad():
        for imgs, img_paths in tqdm(loader, desc="Predicting Test Set"):
            imgs = imgs.to(device, non_blocking=True)
            
            if PREDICT_TTA:
                probs = _tta_predict(model, imgs, device, n_views=6).cpu().numpy()
            else:
                logits, _ = model(imgs)
                probs = torch.sigmoid(logits).cpu().numpy()
                
            for path, prob in zip(img_paths, probs):
                image_ids.append(Path(path).name)
                all_probs.append(float(prob))

    # Apply threshold to generate binary targets
    predictions = [1 if p >= best_thresh else 0 for p in all_probs]

    df = pd.DataFrame({
        "image_id": image_ids,
        "target": predictions
    })
    
    df.to_csv(SUBMISSION_OUT, index=False)

    n_good = predictions.count(0)
    n_fault = predictions.count(1)
    
    print(f"\n[DONE] Predictions saved to: {SUBMISSION_OUT}")
    print(f"       Good Rims (0): {n_good}  |  Faulty Rims (1): {n_fault}")
    print(f"       Using Threshold: {best_thresh:.2f}")

if __name__ == "__main__":
    generate_submission()


[INFO] Starting Inference pipeline on cuda
[INFO] Found 4418 test images.
[INFO] EfficientNet-B3 : 1536ch
[INFO] RegNetY-8GF     : 2016ch
[INFO] ConvNeXt-Small  : 768ch


Predicting Test Set: 100%|██████████| 277/277 [57:45<00:00, 12.51s/it]


[DONE] Predictions saved to: /kaggle/working/submission.csv
       Good Rims (0): 1775  |  Faulty Rims (1): 2643
       Using Threshold: 0.43
